# BERTLens: Step-by-Step Colab Run

This notebook reproduces tokenization inspection, attention visualization, contextual polysemy analysis, and layer-wise probing from the BERTLens repository. Run cells in order. The first model-backed cell downloads `bert-base-uncased`.

In [ ]:
# 1. Clone the project
!git clone https://github.com/arpitpaliwal007/BERTLens.git
%cd BERTLens
!git log -1 --oneline

In [ ]:
# 2. Install project dependencies
!pip install -q -e .
!bertlens --help

## 1. Inspect WordPiece tokenization
The target word is aligned through character offsets. This remains correct even when a word splits into several WordPieces.

In [ ]:
!bertlens tokenize --text "The bank approved the loan." --target-word bank --output reports/tokenization.json --device cpu
!cat reports/tokenization.json

## 2. Visualize one attention head
Layer indices for the attention module run from 0 to 11; head indices run from 0 to 11 for `bert-base-uncased`.

In [ ]:
!bertlens attention --text "The bank approved the loan." --layer 8 --head 3 --output reports/attention_layer8_head3.png --device cpu
from IPython.display import Image, display
display(Image(filename='reports/attention_layer8_head3.png'))

## 3. Analyze contextual word senses
The included dataset contrasts financial and river-bank uses. We pool the target word's subword embeddings, cluster the contexts, and compare the clusters against supplied gold sense labels.

In [ ]:
!bertlens senses --input data/polysemy_examples.csv --target-word bank --layers 0,4,8,12 --output-dir reports/polysemy --device cpu
!cat reports/polysemy/metrics.json

In [ ]:
from pathlib import Path
for path in sorted(Path('reports/polysemy').glob('*_pca.png')):
    print(path.name)
    display(Image(filename=str(path)))

## 4. Run the layer-wise control probe
This intentionally simple short-versus-long sentence probe checks the extraction and evaluation pipeline. It is not a syntax benchmark.

In [ ]:
!bertlens probe --input data/probe_length_examples.csv --layers 0,4,8,12 --output-dir reports/probe --device cpu
!cat reports/probe/metrics.json
display(Image(filename='reports/probe/accuracy_by_layer.png'))

## Next experiments
- Replace `data/polysemy_examples.csv` with your own `sentence,target_word,sense` dataset.
- Try `--layers all` for a complete layer sweep.
- Change `--model` to another compatible BERT-style Hugging Face model.
- Use a GPU runtime for larger context sets.